**데이터 전처리(Data Preprocessing) 개념 정리**

**1. 데이터 전처리 정의 및 목적**

- **정의**: 원시 데이터(raw data)를 분석 및 머신러닝에 적합한 형태 변환
- **목적**: 데이터 품질 향상, 분석/모델링 과정 오류 최소화 및 정확한 결과 도출

**2. 데이터 전처리  작업**

- **정제(Cleansing)**
    - **결측치 처리**:
        - 개념: 누락된 값(Missing values) 식별 및 삭제/대체
        - *식별*: 단순 탐색, 시각화, 결측치 패턴 분석
        - *처리*: 삭제, 평균/중앙값/최빈값 대체, 전진/후진 채우기, 임의 값, KNN/회귀/다중 대체
    - **이상치 처리**:
        - 개념 :  다른 데이터 포인트와 현저히 차이 나는 값 처리
        - *식별*: IQR 방법, 박스플롯(Boxplot) 사용, 상한선/하한선 설정
        - *처리*: 이상치 제거, 평균/중앙값/최빈값 대체, 로그/제곱근 변환, 클리핑(Clipping), 회귀 모델/KNN 대체
    - **중복 데이터 제거**: 동일 데이터 레코드 정리
- **통합(Integration)**:
    - 개념: 분석을 위한 여러 데이터원 통합
- **변환(Transformation)**
    - 데이터 분포 정상화, 범위 표준화, 이상치 영향 감소 등을 목적으로 다루기 쉬운 형식 변환
    - 척도 맞추기 작업, 스케일링, 정규화, 범주형 데이터 인코딩
- **데이터 축소(Reduction)**: 차원(변수) 축소, 샘플링
- **특징 선택 및 생성(Feature Selection & Engineering)**: 모델 성능 향상을 위한 주요 특징(변수) 선택 및 생성

Seaborn의 `titanic` 데이터셋과 Scikit-learn 라이브러리를 활용해 전처리 전 과정을 구현한 예시 코드입니다.

## 1. titanic 데이터셋(seaborn)
- 전처리: 중복제거, IQR 기준 클리핑(이상치), StandardScaler 스케일링, PCA차원 축소

In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 데이터 로드
df = sns.load_dataset("titanic")

# ==========================================
# 1. 정제 (Cleansing)
# ==========================================

# 1-1. 결측치 처리
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# 1-2. 이상치 처리 (IQR 클리핑)
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
df["fare"] = np.clip(df["fare"], lower_bound, upper_bound)

# 1-3. 중복 데이터 제거 및 인덱스 재정렬 (중요: 인덱스 불일치 방지)
df = df.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. 변환 (Transformation)
# ==========================================

# 2-1. 범주형 데이터 인코딩 (One-Hot Encoding)
encoder = OneHotEncoder(sparse_output=False, drop="first")
encoded_sex = encoder.fit_transform(df[["sex"]])
encoded_sex_df = pd.DataFrame(
    encoded_sex, columns=encoder.get_feature_names_out(["sex"])
)

# 2-2. 스케일링 및 정규화
scaler = StandardScaler()
df[["age_scaled", "fare_scaled"]] = scaler.fit_transform(df[["age", "fare"]])


# ==========================================
# 3. 특징 선택 및 생성 (Feature Engineering)
# ==========================================

# 파생 변수 생성
df["family_size"] = df["sibsp"] + df["parch"] + 1

# 전처리 완료된 특징 및 타겟 결합
X_features = pd.concat(
    [df[["age_scaled", "fare_scaled", "family_size"]], encoded_sex_df], axis=1
)
y = df["survived"]


# ==========================================
# 4. 데이터 축소 및 분할 (Reduction & Split)
# ==========================================

# Train / Test 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y, test_size=0.2, random_state=42
)

# 차원 축소 (PCA)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("전처리 완료 데이터 크기:", X_features.shape)
print("PCA 축소 후 Train 데이터 크기:", X_train_pca.shape)


# ==========================================
# 5. 모델 학습 & 평가 (LogisticRegression)
# ==========================================

# 최신 scikit-learn 기준 L2 규제 적용 (l1_ratio=0.0)
model = LogisticRegression(solver="saga", l1_ratio=0.0, random_state=42)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

print("\n--- Titanic Classification Report ---")
print(classification_report(y_test, y_pred))


# ==========================================
# 6. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "titanic_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

전처리 완료 데이터 크기: (778, 4)
PCA 축소 후 Train 데이터 크기: (622, 2)

--- Titanic Classification Report ---
              precision    recall  f1-score   support

           0       0.58      0.94      0.72        86
           1       0.71      0.17      0.28        70

    accuracy                           0.60       156
   macro avg       0.64      0.56      0.50       156
weighted avg       0.64      0.60      0.52       156

모델 저장 완료: model\titanic_logistic_model.pkl


Train / Validation / Test (60:20:20) 3개 세트로 분할하고, Validation 검증 ➔ 모델 저장 ➔ 모델 불러오기 ➔ Test 최종 평가

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 데이터 로드
df = sns.load_dataset("titanic")

# ==========================================
# 1. 기초 정제 (Cleansing & Feature)
# ==========================================
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr
df["fare"] = np.clip(df["fare"], lower_bound, upper_bound)

df["family_size"] = df["sibsp"] + df["parch"] + 1
df = df.drop_duplicates().reset_index(drop=True)


# ==========================================
# 2. 원본 데이터 상태에서 먼저 분할 (Data Leakage 방지)
# ==========================================
X_raw = df[["sex", "age", "fare", "family_size"]]
y = df["survived"]

# 1차 분할: Train+Val (80%) / Test (20%)
X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 2차 분할: Train (60%) / Validation (20%)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)


# ==========================================
# 3. Train 데이터 기준으로만 fit 수행 및 각각 transform
# ==========================================

# 3-1. OneHotEncoder (fit: Train 전용)
encoder = OneHotEncoder(sparse_output=False, drop="first")
sex_train_enc = pd.DataFrame(encoder.fit_transform(X_train_raw[["sex"]]), columns=encoder.get_feature_names_out(["sex"]), index=X_train_raw.index)
sex_val_enc = pd.DataFrame(encoder.transform(X_val_raw[["sex"]]), columns=encoder.get_feature_names_out(["sex"]), index=X_val_raw.index)

# 3-2. StandardScaler (fit: Train 전용)
scaler = StandardScaler()
num_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw[["age", "fare"]]), columns=["age_scaled", "fare_scaled"], index=X_train_raw.index)
num_val_scaled = pd.DataFrame(scaler.transform(X_val_raw[["age", "fare"]]), columns=["age_scaled", "fare_scaled"], index=X_val_raw.index)

# 전처리 결과 결합
X_train = pd.concat([num_train_scaled, X_train_raw[["family_size"]], sex_train_enc], axis=1)
X_val = pd.concat([num_val_scaled, X_val_raw[["family_size"]], sex_val_enc], axis=1)

# 3-3. PCA (fit: Train 전용)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)


# ==========================================
# 4. 모델 학습 & Validation 테스트
# ==========================================
model = LogisticRegression(solver="saga", l1_ratio=0.0, random_state=42)
model.fit(X_train_pca, y_train)

y_val_pred = model.predict(X_val_pca)
print("--- [Validation] Performance ---")
print(classification_report(y_val, y_val_pred))


# ==========================================
# 5. 전처리기(Scaler, Encoder, PCA) & 모델 한번에 저장
# ==========================================
save_dir = "model"
os.makedirs(save_dir, exist_ok=True)

# 저장할 객체들을 딕셔너리로 패키징
artifacts = {
    "encoder": encoder,
    "scaler": scaler,
    "pca": pca,
    "model": model
}

artifacts_path = os.path.join(save_dir, "titanic_pipeline.pkl")
joblib.dump(artifacts, artifacts_path)
print(f"\n전처리기 및 모델 저장 완료: {artifacts_path}")


# ==========================================
# 6. 저장된 아티팩트 불러오기 후 Test 데이터에 적용 및 실행
# ==========================================

# 1) 저장된 아티팩트 로드
loaded_artifacts = joblib.load(artifacts_path)

loaded_encoder = loaded_artifacts["encoder"]
loaded_scaler = loaded_artifacts["scaler"]
loaded_pca = loaded_artifacts["pca"]
loaded_model = loaded_artifacts["model"]

# 2) 불러온 Scaler 및 Encoder로 원본 Test 데이터(X_test_raw) 변환
test_sex_enc = pd.DataFrame(
    loaded_encoder.transform(X_test_raw[["sex"]]),
    columns=loaded_encoder.get_feature_names_out(["sex"]),
    index=X_test_raw.index
)
test_num_scaled = pd.DataFrame(
    loaded_scaler.transform(X_test_raw[["age", "fare"]]),
    columns=["age_scaled", "fare_scaled"],
    index=X_test_raw.index
)
X_test_prepared = pd.concat([test_num_scaled, X_test_raw[["family_size"]], test_sex_enc], axis=1)

# 3) 불러온 PCA 적용
X_test_pca = loaded_pca.transform(X_test_prepared)

# 4) 불러온 모델로 Test 예측
y_test_pred = loaded_model.predict(X_test_pca)

print("\n--- [Loaded Model - Test Data] Performance ---")
print(classification_report(y_test, y_test_pred))

--- [Validation] Performance ---
              precision    recall  f1-score   support

           0       0.64      0.85      0.73        92
           1       0.59      0.31      0.41        64

    accuracy                           0.63       156
   macro avg       0.61      0.58      0.57       156
weighted avg       0.62      0.63      0.60       156


전처리기 및 모델 저장 완료: model\titanic_pipeline.pkl

--- [Loaded Model - Test Data] Performance ---
              precision    recall  f1-score   support

           0       0.60      0.71      0.65        92
           1       0.44      0.33      0.38        64

    accuracy                           0.55       156
   macro avg       0.52      0.52      0.51       156
weighted avg       0.53      0.55      0.54       156



## 2. Penguins 데이터셋 (seaborn)

- **전처리:** 결측치 대체(KNNImputer/최빈값), 범주형 원핫인코딩, IQR 상하한선제거, RobustScaler

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd  # get_dummies 사용을 위해 추가
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. 데이터 로드 (결측치 및 범주형 데이터 포함)
df = sns.load_dataset('penguins')
df = df.dropna(subset=['species'])
df['target'] = (df['species'] == 'Adelie').astype(int)

# 2. 전처리 (범주형 데이터 인코딩 & 통합) - pd.get_dummies로 수정
df = pd.get_dummies(df, columns=['island', 'sex'], drop_first=True)

num_cols = [
    'bill_length_mm',
    'bill_depth_mm',
    'flipper_length_mm',
    'body_mass_g',
]

# 3. 전처리 (결측치 처리: KNNImputer)
imputer = KNNImputer(n_neighbors=5)
df[num_cols] = imputer.fit_transform(df[num_cols])

# 4. 전처리 (이상치 제거: IQR 하한/상한)
for col in num_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

X = df.drop(columns=['species', 'target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. 변환 (스케일링: RobustScaler)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. 모델 학습 & 평가 (최신 scikit-learn 규격 적용)
model = LogisticRegression(max_iter=1000,solver='saga', l1_ratio=0.0, random_state=42)
model.fit(X_train_scaled, y_train)

print("--- Penguins Classification Report ---")
print(classification_report(y_test, model.predict(X_test_scaled)))

# 7. 폴더 확인 후 모델 저장

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "penguins_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

## 3. Breast Cancer 데이터셋 (scikit-learn)

- **전처리:** 로그 변환(이상치 완화), SelectKBest(특징 선택), MinMaxScaler

In [ ]:

import os  # os 모듈 추가
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 로드
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

# 2. 전처리 (이상치 완화: 로그 변환 - 편향된 분포 정상화)
skewed_cols = [col for col in X.columns if X[col].skew() > 1.0]
for col in skewed_cols:
  X[col] = np.log1p(X[col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. 변환 (스케일링: MinMaxScaler)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. 특징 선택 (Feature Selection: SelectKBest)
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# 5. 모델 학습 & 평가 (최신 scikit-learn 규격 반영)
model = LogisticRegression(max_iter=1000,solver='saga', l1_ratio=0.0, random_state=42)
model.fit(X_train_selected, y_train)

print("--- Breast Cancer Classification Report ---")
print(classification_report(y_test, model.predict(X_test_selected)))

# ==========================================
# 6. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "cancer_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")


## 4. Diamonds 데이터셋 (seaborn)

- **전처리:** 순서형 인코딩(Ordinal), 제곱근 변환, 과도한 샘플링 축소(Downsampling)

In [ ]:
import os  # os 모듈 추가
import joblib
import numpy as np
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# 1. 데이터 로드
df = sns.load_dataset('diamonds')

# 2. 데이터 축소 (샘플링: 연산속도 최적화를 위한 언더샘플링)
df = df.sample(n=5000, random_state=42)

# 타겟 설정: 가격 3000달러 이상 여부 (이진분류)
df['is_expensive'] = (df['price'] > 3000).astype(int)

# 3. 범주형 데이터 인코딩 (Ordinal Encoding)
ordinal_cols = ['cut', 'color', 'clarity']
encoder = OrdinalEncoder()
df[ordinal_cols] = encoder.fit_transform(df[ordinal_cols])

# 4. 이상치 처리 (제곱근 변환: 왜도 감소)
df['carat_sqrt'] = np.sqrt(df['carat'])

X = df.drop(columns=['price', 'is_expensive', 'carat'])
y = df['is_expensive']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. 모델 학습 & 평가 (최신 scikit-learn 규격 반영)
model = LogisticRegression(max_iter=1000,solver='saga', l1_ratio=0.0, random_state=42)
model.fit(X_train_scaled, y_train)

print("--- Diamonds Classification Report ---")
print(classification_report(y_test, model.predict(X_test_scaled)))

# ==========================================
# 7. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "diamonds_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

## 5. Wine 데이터셋 (scikit-learn)

- **전처리:** 결측치 강제 생성 후 SimpleImputer(평균값 대체), 파생 변수 생성, RFE 특징 선택

In [5]:
import os  # os 모듈 추가
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 데이터 로드 및 이진 분류 변환
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = (wine.target == 0).astype(int)

# 2. 임의로 결측치 생성 (실습용)
np.random.seed(42)
X.iloc[0:10, 0] = np.nan

# 3. 결측치 처리 (SimpleImputer: 평균값 대체)
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# 4. 특징 생성 (Feature Generation: 파생 변수 추가)
X_imputed['alc_flava_ratio'] = (
    X_imputed['alcohol'] / (X_imputed['flavanoids'] + 1e-5)
)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)

# 5. 변환 (스케일링)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. 특징 선택 (RFE: 재귀적 특징 제거)
estimator = LogisticRegression(solver='saga', l1_ratio=0.0, random_state=42)
selector = RFE(estimator, n_features_to_select=5)
X_train_rfe = selector.fit_transform(X_train_scaled, y_train)
X_test_rfe = selector.transform(X_test_scaled)

# 7. 모델 학습 및 평가 (최신 scikit-learn 규격 반영)
model = LogisticRegression(max_iter=1000, solver='saga', l1_ratio=0.0, random_state=42)
model.fit(X_train_rfe, y_train)

print("--- Wine Classification Report ---")
print(classification_report(y_test, model.predict(X_test_rfe)))

# ==========================================
# 8. 폴더 확인 후 모델 저장
# ==========================================

save_dir = "model"
if not os.path.exists(save_dir):
  os.makedirs(save_dir)

model_path = os.path.join(save_dir, "wine_logistic_model.pkl")
joblib.dump(model, model_path)
print(f"모델 저장 완료: {model_path}")

--- Wine Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        22
           1       1.00      1.00      1.00        14

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

모델 저장 완료: model\wine_logistic_model.pkl


c:\sk-encoa\data-collection-workspace\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
